# Revenue Forecasting Demo


In [1]:
import math
import os
import sys
from pathlib import Path

import mlflow
import mlflow.lightgbm
import mlflow.pytorch
import mlflow.sklearn
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch
from dotenv import load_dotenv
from mlflow.tracking import MlflowClient
from sklearn.metrics import mean_absolute_error, mean_squared_error

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")

minio_endpoint = os.getenv("MINIO_ENDPOINT")
if minio_endpoint:
    os.environ["MLFLOW_S3_ENDPOINT_URL"] = f"http://{minio_endpoint}"

minio_access_key = os.getenv("MINIO_ACCESS_KEY")
if minio_access_key:
    os.environ["AWS_ACCESS_KEY_ID"] = minio_access_key

minio_secret_key = os.getenv("MINIO_SECRET_KEY")
if minio_secret_key:
    os.environ["AWS_SECRET_ACCESS_KEY"] = minio_secret_key

os.environ["MLFLOW_S3_IGNORE_TLS"] = "true"

tracking_uri = os.getenv("MLFLOW_TRACKING_URI")
if tracking_uri:
    mlflow.set_tracking_uri(tracking_uri)


def week_start_dates(frame: pd.DataFrame) -> pd.Series:
    return frame.apply(
        lambda row: pd.Timestamp.fromisocalendar(
            int(row["year"]),
            int(row["week_of_year"]),
            1,
        ),
        axis=1,
    )


def regression_metrics(y_true, y_pred) -> tuple[float, float]:
    rmse = math.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    return rmse, mae


print("Setup hoàn tất.")


c:\Users\vanthang\anaconda3\envs\dlh\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
c:\Users\vanthang\anaconda3\envs\dlh\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Setup hoàn tất.


## 1. Import helper và cấu hình chung


In [2]:
from src.mlops.data_loader import load_revenue_data
from src.mlops.revenue.features import TARGET_COL, TRAIN_RATIO, build_lag_features
from src.mlops.revenue.trainer_prophet import PROPHET_REGRESSORS, _build_prophet_df

client = MlflowClient()


def get_latest_model_version(model_name: str) -> tuple[str, str]:
    versions = client.search_model_versions(f"name='{model_name}'")
    if not versions:
        raise ValueError(f"Không tìm thấy model trong MLflow Registry: {model_name}")

    latest = max(versions, key=lambda version: int(version.version))
    print(f"{model_name}: version={latest.version} | run_id={latest.run_id[:8]}...")
    return latest.version, latest.run_id


print("Import hoàn tất.")


Import hoàn tất.


## 2. Load dữ liệu


In [3]:
df = (
    load_revenue_data()
    .sort_values(["year", "week_of_year"])
    .reset_index(drop=True)
)
df["time_label"] = (
    df["year"].astype(str)
    + "-W"
    + df["week_of_year"].astype(str).str.zfill(2)
)

print(f"{len(df)} tuần | {df['time_label'].iloc[0]} -> {df['time_label'].iloc[-1]}")
df.tail(3)


90 tuần | 2016-W40 -> 2018-W35


,year,week_of_year,month,weekly_revenue,order_count,total_product_value,total_freight_value,avg_order_value,avg_items_per_order,freight_revenue_ratio,...,order_count_lag_1,order_count_lag_2,order_count_lag_4,rolling_avg_4w,rolling_std_4w,rolling_avg_8w,order_rolling_avg_4w,revenue_momentum,revenue_acceleration,time_label
87,2018,33,8,285766.68,1839,243458.02,42278.27,155.392431,1.104948,0.147947,...,1931.0,2002.0,1649.0,294014.5375,30571.072554,248180.32125,1798.0,-0.004697,-0.244712,2018-W33
88,2018,34,8,145680.87,1054,125412.50,20342.21,138.217144,1.126186,0.139635,...,1839.0,1931.0,1610.0,300899.5875,21613.251113,255767.32000,1845.5,-0.103338,-0.245936,2018-W34
89,2018,35,8,11995.53,116,10565.86,1429.67,103.409741,1.137931,0.119184,...,1054.0,1839.0,2002.0,267588.2350,82810.701944,246601.27750,1706.5,-0.490210,-0.382660,2018-W35


## 3. Load model và scaler từ MLflow Registry


In [8]:
lgb_version, lgb_run_id = get_latest_model_version("revenue_lightgbm")
lstm_version, lstm_run_id = get_latest_model_version("revenue_lstm")
prophet_version, prophet_run_id = get_latest_model_version("revenue_prophet")

lgb_model = mlflow.lightgbm.load_model(f"models:/revenue_lightgbm/{lgb_version}")
lstm_model = mlflow.pytorch.load_model(f"models:/revenue_lstm/{lstm_version}")
prophet_model = mlflow.prophet.load_model(f"models:/revenue_prophet/{prophet_version}")
lstm_model.eval()

lgb_scaler = mlflow.sklearn.load_model(f"runs:/{lgb_run_id}/scaler")
feat_scaler = mlflow.sklearn.load_model(f"runs:/{lstm_run_id}/feat_scaler")
target_scaler = mlflow.sklearn.load_model(f"runs:/{lstm_run_id}/target_scaler")

print("Load model và scaler thành công.")

revenue_lightgbm: version=1 | run_id=015ff61e...
revenue_lstm: version=1 | run_id=05cff759...
revenue_prophet: version=1 | run_id=f74de930...


Load model và scaler thành công.


## 4. Inference LightGBM


In [5]:
df_lag = build_lag_features(df.copy())
split_lgb = int(len(df_lag) * TRAIN_RATIO)

if hasattr(lgb_scaler, "feature_names_in_"):
    lgb_features = list(lgb_scaler.feature_names_in_)
else:
    exclude_cols = {
        "year",
        "week_of_year",
        TARGET_COL,
        "total_product_value",
        "total_freight_value",
    }
    lgb_features = [
        col
        for col in df_lag.columns
        if col not in exclude_cols
        and df_lag[col].dtype in ("float64", "int64", "float32", "int32")
    ]

missing_features = [col for col in lgb_features if col not in df_lag.columns]
if missing_features:
    raise ValueError(f"Thiếu feature LightGBM: {missing_features}")

X_val_lgb = df_lag[lgb_features].iloc[split_lgb:]
y_val_lgb = df_lag[TARGET_COL].iloc[split_lgb:].to_numpy()

X_val_lgb_scaled = lgb_scaler.transform(X_val_lgb)
lgb_preds = np.maximum(np.expm1(lgb_model.predict(X_val_lgb_scaled)), 0)

rmse_lgb, mae_lgb = regression_metrics(y_val_lgb, lgb_preds)
lgb_results = pd.DataFrame({
    "ds": week_start_dates(df_lag.iloc[split_lgb:]).to_numpy(),
    "actual": y_val_lgb,
    "lightgbm": lgb_preds,
})

print(f"LightGBM -> RMSE: {rmse_lgb:,.2f} | MAE: {mae_lgb:,.2f}")


LightGBM -> RMSE: 23,296.40 | MAE: 19,983.96


## 5. Inference LSTM


In [6]:
df_seq = build_lag_features(df.copy())
lstm_features = list(feat_scaler.feature_names_in_)

missing_features = [col for col in lstm_features if col not in df_seq.columns]
if missing_features:
    raise ValueError(f"Thiếu feature LSTM: {missing_features}")

run_params = client.get_run(lstm_run_id).data.params
seq_len = int(run_params.get("seq_len", 4))

X_raw = feat_scaler.transform(df_seq[lstm_features])
target_log = np.log1p(df_seq[[TARGET_COL]].to_numpy())
y_raw = target_scaler.transform(target_log)

X_seq, y_seq = [], []
for i in range(len(df_seq) - seq_len):
    X_seq.append(X_raw[i : i + seq_len])
    y_seq.append(y_raw[i + seq_len])

X_seq = np.array(X_seq, dtype=np.float32)
y_seq = np.array(y_seq, dtype=np.float32)

split_lstm = int(len(X_seq) * TRAIN_RATIO)
X_val_lstm = torch.tensor(X_seq[split_lstm:], dtype=torch.float32)
y_val_lstm = y_seq[split_lstm:]

device = next(lstm_model.parameters()).device
lstm_model.eval()

with torch.no_grad():
    preds_norm = lstm_model(X_val_lstm.to(device)).cpu().numpy()

preds_log = target_scaler.inverse_transform(preds_norm)
truth_log = target_scaler.inverse_transform(y_val_lstm)

lstm_preds = np.maximum(np.expm1(preds_log), 0).flatten()
y_val_lstm_orig = np.maximum(np.expm1(truth_log), 0).flatten()

lstm_results = pd.DataFrame({
    "ds": week_start_dates(df_seq.iloc[seq_len + split_lstm :]).to_numpy(),
    "actual": y_val_lstm_orig,
    "lstm": lstm_preds,
})

rmse_lstm, mae_lstm = regression_metrics(y_val_lstm_orig, lstm_preds)
print(f"LSTM     -> RMSE: {rmse_lstm:,.2f} | MAE: {mae_lstm:,.2f}")


LSTM     -> RMSE: 67,163.50 | MAE: 56,844.21


## 6. Inference Prophet


In [9]:
df_prophet = _build_prophet_df(df.copy())
split_prophet = int(len(df_prophet) * TRAIN_RATIO)
val_prophet = df_prophet.iloc[split_prophet:].copy()

regressors_used = [col for col in PROPHET_REGRESSORS if col in df_prophet.columns]
forecast = prophet_model.predict(val_prophet[["ds"] + regressors_used])

prophet_preds = np.maximum(forecast["yhat"].to_numpy(), 0)
prophet_truth = val_prophet["y"].to_numpy()

rmse_prophet, mae_prophet = regression_metrics(prophet_truth, prophet_preds)
prophet_results = pd.DataFrame({
    "ds": val_prophet["ds"].to_numpy(),
    "actual": prophet_truth,
    "prophet": prophet_preds,
})

print(f"Prophet  -> RMSE: {rmse_prophet:,.2f} | MAE: {mae_prophet:,.2f}")

Prophet  -> RMSE: 138,540.86 | MAE: 116,371.15


## 7. Ensemble LightGBM + Prophet


In [10]:
PROPHET_WEIGHT = 0.4
LGB_WEIGHT = 1.0 - PROPHET_WEIGHT

ensemble_base = lgb_results.merge(
    prophet_results[["ds", "prophet"]],
    on="ds",
    how="inner",
)
ensemble_base["ensemble"] = (
    LGB_WEIGHT * ensemble_base["lightgbm"]
    + PROPHET_WEIGHT * ensemble_base["prophet"]
)

ensemble_truth = ensemble_base["actual"].to_numpy()
ensemble_preds = ensemble_base["ensemble"].to_numpy()
rmse_ensemble, mae_ensemble = regression_metrics(ensemble_truth, ensemble_preds)

ensemble_results = ensemble_base[["ds", "actual", "ensemble"]].copy()
print(f"Ensemble -> RMSE: {rmse_ensemble:,.2f} | MAE: {mae_ensemble:,.2f}")


Ensemble -> RMSE: 62,264.86 | MAE: 53,288.05


## 8. Biểu đồ so sánh model


In [11]:
plot_df = (
    lgb_results.rename(columns={"actual": "actual_lgb"})
    .merge(
        lstm_results.rename(columns={"actual": "actual_lstm"}),
        on="ds",
        how="outer",
    )
    .merge(
        prophet_results.rename(columns={"actual": "actual_prophet"}),
        on="ds",
        how="outer",
    )
    .merge(ensemble_results[["ds", "ensemble"]], on="ds", how="outer")
    .sort_values("ds")
)
plot_df["actual"] = (
    plot_df["actual_lgb"]
    .combine_first(plot_df["actual_prophet"])
    .combine_first(plot_df["actual_lstm"])
)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=plot_df["ds"],
    y=plot_df["actual"],
    name="Actual",
    line=dict(color="#00d4ff", width=2),
))
fig.add_trace(go.Scatter(
    x=plot_df["ds"],
    y=plot_df["lightgbm"],
    name=f"LightGBM (RMSE={rmse_lgb:,.0f})",
    line=dict(color="#ff6b35", width=2, dash="dot"),
))
fig.add_trace(go.Scatter(
    x=plot_df["ds"],
    y=plot_df["lstm"],
    name=f"LSTM (RMSE={rmse_lstm:,.0f})",
    line=dict(color="#b388ff", width=2, dash="dashdot"),
))
fig.add_trace(go.Scatter(
    x=plot_df["ds"],
    y=plot_df["prophet"],
    name=f"Prophet (RMSE={rmse_prophet:,.0f})",
    line=dict(color="#a8ff78", width=2, dash="dash"),
))
fig.add_trace(go.Scatter(
    x=plot_df["ds"],
    y=plot_df["ensemble"],
    name=f"Ensemble (RMSE={rmse_ensemble:,.0f})",
    line=dict(color="#f7971e", width=3),
))

fig.update_layout(
    title="Revenue Forecasting - Model Comparison",
    xaxis_title="Week",
    yaxis_title="Revenue (BRL)",
    template="plotly_dark",
    height=500,
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
    hovermode="x unified",
)
fig.show()


## 9. Bảng so sánh RMSE/MAE


In [12]:
summary = pd.DataFrame([
    {"Model": "LightGBM", "RMSE": rmse_lgb, "MAE": mae_lgb},
    {"Model": "LSTM", "RMSE": rmse_lstm, "MAE": mae_lstm},
    {"Model": "Prophet", "RMSE": rmse_prophet, "MAE": mae_prophet},
    {"Model": "Ensemble (LGB + Prophet)", "RMSE": rmse_ensemble, "MAE": mae_ensemble},
])

best_idx = summary["RMSE"].idxmin()
summary["Champion"] = ""
summary.loc[best_idx, "Champion"] = "Champion"

summary = summary.sort_values("RMSE").reset_index(drop=True)
summary["RMSE"] = summary["RMSE"].map("{:,.2f}".format)
summary["MAE"] = summary["MAE"].map("{:,.2f}".format)
summary


,Model,RMSE,MAE,Champion
0,LightGBM,"23,296.40","19,983.96",Champion
1,Ensemble (LGB + Prophet),"62,264.86","53,288.05",
2,LSTM,"67,163.50","56,844.21",
3,Prophet,"138,540.86","116,371.15",
